# Dynamic Withdrawal Strategies - Step-by-Step Test

이 노트북은 Dynamic 전략 구현을 단계별로 테스트합니다.

## Step 1: 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

In [ ]:
# 벤치마크 데이터 로드
with open('benchmark_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   데이터 크기: {data.shape}")
print(f"   날짜 범위: {data.index[0].date()} ~ {data.index[-1].date()}")
print(f"\n컬럼:\n{data.columns.tolist()}")

## Step 2: DataPreprocessor 테스트

In [ ]:
from withdrawal_backtest import DataPreprocessor, PORTFOLIOS

print(f"DataPreprocessor 임포트 성공")
print(f"\n포트폴리오 목록:")
for i, port_name in enumerate(PORTFOLIOS.keys(), 1):
    port = PORTFOLIOS[port_name]
    print(f"  {i}. {port_name}: 목표 수익률={port['target_return']:.1f}%, 목표 변동성={port['target_risk']:.2f}%")

In [ ]:
# DataPreprocessor 실행
try:
    preprocessor = DataPreprocessor(data, add_portfolios=True)
    returns_df, month_starts = preprocessor.get_data()
    
    print(f"✅ DataPreprocessor 완료")
    print(f"   Returns DataFrame: {returns_df.shape}")
    print(f"   Month Starts Series: {month_starts.shape}")
    print(f"\n   Returns 컬럼:")
    print(f"   {returns_df.columns.tolist()[:10]}...")
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Dynamic Simulator 테스트

In [ ]:
from dynamic_simulator import GuardrailsWithdrawal, GuytonKlingerWithdrawal, DynamicWithdrawalSimulator

print("✅ Dynamic Simulator 클래스 임포트 성공")

# DynamicWithdrawalSimulator 생성
simulator = DynamicWithdrawalSimulator(returns_df, month_starts)
print(f"✅ DynamicWithdrawalSimulator 생성 완료")
print(f"   총 날짜: {len(simulator.dates)}")
print(f"   월초 개수: {np.sum(month_starts)}")

## Step 4: Guardrails 백테스트 (간단한 테스트)

In [ ]:
# 하나의 포트폴리오와 인출률로 빠른 테스트
test_portfolio = 'Port_5.0%'
test_wr = 0.05  # 5%
horizon_years = 10

print(f"테스트 설정:")
print(f"  포트폴리오: {test_portfolio}")
print(f"  인출률: {test_wr*100:.1f}%")
print(f"  기간: {horizon_years}년")

try:
    results_df = simulator.run_guardrails_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guardrails 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {results_df.shape}")
    print(f"  컬럼: {results_df.columns.tolist()}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 5: 메트릭 계산 테스트

In [ ]:
from metrics_calculator import MetricsCalculator

metrics_calc = MetricsCalculator()

try:
    metrics = metrics_calc.calculate_optimization_metrics(results_df, v0=100.0)
    
    print(f"✅ 메트릭 계산 완료")
    print(f"\n주요 메트릭:")
    print(f"  총 인출액 (평균): {metrics['total_withdrawal_mean']:,.2f}")
    print(f"  총 인출액 (범위): {metrics['total_withdrawal_worst']:,.2f} ~ {metrics['total_withdrawal_best']:,.2f}")
    print(f"  YoY 변동성 (평균): {metrics['yoy_volatility_mean']:.4f}")
    print(f"  YoY 변동성 (90분위): {metrics['yoy_volatility_90pct']:.4f}")
    print(f"  실패율: {metrics['failure_rate']:.2%}")
    print(f"  최종 NAV (중앙값): {metrics['terminal_nav_median']:.2%}")
    print(f"  경로 수: {metrics['total_paths']}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Guyton-Klinger 백테스트

In [ ]:
try:
    gk_results_df = simulator.run_guyton_klinger_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guyton-Klinger 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {gk_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(gk_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 7: WithdrawalOptimizer 테스트

In [ ]:
from optimizer import WithdrawalOptimizer

optimizer = WithdrawalOptimizer(returns_df, month_starts)

print(f"✅ WithdrawalOptimizer 생성 완료")

# 작은 범위로 빠른 테스트
withdrawal_rates = np.array([0.04, 0.05, 0.06])
constraints = {
    'max_yoy_volatility': 0.20,
    'max_failure_rate': 0.10,
    'min_terminal_nav': 0.40
}

print(f"\n테스트 설정:")
print(f"  인출률: {withdrawal_rates}")
print(f"  제약조건:")
print(f"    - Max YoY Volatility: {constraints['max_yoy_volatility']:.0%}")
print(f"    - Max Failure Rate: {constraints['max_failure_rate']:.0%}")
print(f"    - Min Terminal NAV: {constraints['min_terminal_nav']:.0%}")

In [ ]:
try:
    # 단일 포트폴리오 최적화 (빠른 테스트)
    portfolio_results = optimizer.optimize_single_portfolio(
        portfolio_name='Port_5.0%',
        withdrawal_rates=withdrawal_rates,
        horizon_years=10,
        constraints=constraints,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ 단일 포트폴리오 최적화 완료")
    print(f"\n결과 구조:")
    for wr, strategies in portfolio_results.items():
        print(f"  인출률 {wr:.2%}:")
        for strategy, metrics in strategies.items():
            feasible = "✓" if metrics.get('is_feasible') else "✗"
            total_w = metrics.get('total_withdrawal_mean', 0)
            print(f"    {strategy:20s} {feasible} - 총 인출액: {total_w:,.2f}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 8: 전체 최적화 (2개 포트폴리오)

In [ ]:
try:
    # 작은 범위로 빠른 테스트
    test_portfolios = ['Port_4.0%', 'Port_5.0%']
    test_wrs = np.array([0.04, 0.05, 0.06])
    
    results_df = optimizer.optimize_all_portfolios(
        portfolio_names=test_portfolios,
        withdrawal_rates=test_wrs,
        horizon_years=10,
        constraints=constraints,
        v0=100.0
    )
    
    print(f"\n✅ 전체 최적화 완료")
    print(f"\n결과 DataFrame: {results_df.shape}")
    print(f"\n전체 결과:")
    display(results_df.head(10))
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 9: 결과 요약

In [ ]:
try:
    print(f"\n최적화 결과 통계:")
    print(f"  전체 시나리오: {len(results_df)}")
    print(f"  가능한 시나리오 (제약 만족): {len(results_df[results_df['Feasible']==True])}")
    print(f"  불가능한 시나리오: {len(results_df[results_df['Feasible']==False])}")
    
    print(f"\n포트폴리오별 최적 솔루션:")
    for portfolio in results_df['Portfolio'].unique():
        df_port = results_df[results_df['Portfolio'] == portfolio]
        for strategy in ['fixed', 'guardrails', 'guyton_klinger']:
            df_strat = df_port[df_port['Strategy'] == strategy]
            df_feasible = df_strat[df_strat['Feasible'] == True]
            if not df_feasible.empty:
                best = df_feasible.loc[df_feasible['Total_Withdrawal'].idxmax()]
                print(f"  {portfolio:12s} {strategy:20s}: WR={best['WR']:.2%}, "
                      f"TotalW={best['Total_Withdrawal']:.0f}, "
                      f"Vol={best['YoY_Volatility']:.2%}")
            else:
                print(f"  {portfolio:12s} {strategy:20s}: No feasible solution")
                
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 10: UI 컴포넌트 테스트

In [ ]:
try:
    from dynamic_strategy_ui import (
        create_pareto_frontier_chart,
        create_strategy_comparison_chart
    )
    
    print(f"✅ UI 컴포넌트 임포트 성공")
    
    # Pareto Frontier 차트 생성
    portfolio_chart = 'Port_5.0%'
    fig = create_pareto_frontier_chart(results_df, portfolio_chart)
    
    print(f"✅ Pareto Frontier 차트 생성 완료")
    print(f"   차트 타입: {type(fig)}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 11: 특정 경로 상세 조회 (get_single_path_detail)

단일 시작일의 일별 경로 데이터를 반환하는 기능 테스트

In [ ]:
# 시작일 선택
test_start_date = '2008-01-02'  # 금융위기 시작
test_portfolio = 'Port_5.0%'
test_strategy = 'guardrails'

print(f"특정 경로 조회:")
print(f"  포트폴리오: {test_portfolio}")
print(f"  시작일: {test_start_date}")
print(f"  전략: {test_strategy}")

try:
    path_df = simulator.get_single_path_detail(
        portfolio=test_portfolio,
        start_date=test_start_date,
        strategy=test_strategy,
        horizon_years=10,
        initial_wr=0.05,
        guardrail_width=0.20,
        inflation_rate=0.02,
        v0=100.0
    )
    
    print(f"\n\u2705 일별 경로 데이터 생성 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {path_df.shape}")
    print(f"  날짜 범위: {path_df['Date'].iloc[0].date()} ~ {path_df['Date'].iloc[-1].date()}")
    print(f"\n컬럼:")
    print(f"  {path_df.columns.tolist()}")
    
    print(f"\n샘플 데이터 (첫 5행):")
    print(path_df.head())
    
    print(f"\n검증:")
    print(f"  Total_NAV = sum(asset NAVs) 확인:")
    path_df['Calculated_Total'] = (
        path_df['NAV_Korean_Equity'] + 
        path_df['NAV_US_Growth'] + 
        path_df['NAV_Bond'] + 
        path_df['NAV_Gold']
    )
    max_diff = abs(path_df['Total_NAV'] - path_df['Calculated_Total']).max()
    print(f"    최대 오차: {max_diff:.10f}")
    
    # 월초 행 수 확인
    n_month_starts = path_df['Is_Month_Start'].sum()
    print(f"  월초 행 수: {n_month_starts} (10년 ≈ 120월)")
    
    # Guardrail 상태 분포
    print(f"  Guardrail 상태 분포:")
    print(path_df['Guardrail_Status'].value_counts())
    
    # 엑셀 저장
    output_path = f'path_{test_start_date}_{test_strategy}.xlsx'
    path_df.drop(columns=['Calculated_Total'], inplace=True)
    path_df.to_excel(output_path, index=False)
    print(f"\n\u2705 엑셀 저장 완료: {output_path}")
    
except Exception as e:
    print(f"\u274c 오류: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Guyton-Klinger 테스트
try:
    path_df_gk = simulator.get_single_path_detail(
        portfolio=test_portfolio,
        start_date=test_start_date,
        strategy='guyton_klinger',
        horizon_years=10,
        initial_wr=0.05,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0
    )
    
    print(f"\u2705 Guyton-Klinger 일별 경로 데이터 생성 완료")
    print(f"  크기: {path_df_gk.shape}")
    print(f"  날짜 범위: {path_df_gk['Date'].iloc[0].date()} ~ {path_df_gk['Date'].iloc[-1].date()}")
    
    # 검증
    path_df_gk['Calculated_Total'] = (
        path_df_gk['NAV_Korean_Equity'] + 
        path_df_gk['NAV_US_Growth'] + 
        path_df_gk['NAV_Bond'] + 
        path_df_gk['NAV_Gold']
    )
    max_diff_gk = abs(path_df_gk['Total_NAV'] - path_df_gk['Calculated_Total']).max()
    print(f"  Total_NAV 최대 오차: {max_diff_gk:.10f}")
    
    print(f"  Guardrail 상태 분포:")
    print(path_df_gk['Guardrail_Status'].value_counts())
    
    # 엑셀 저장
    output_path_gk = f'path_{test_start_date}_guyton_klinger.xlsx'
    path_df_gk.drop(columns=['Calculated_Total'], inplace=True)
    path_df_gk.to_excel(output_path_gk, index=False)
    print(f"\u2705 엑셀 저장 완료: {output_path_gk}")
    
except Exception as e:
    print(f"\u274c 오류: {e}")
    import traceback
    traceback.print_exc()

## 요약

✅ 모든 단계 완료!

- Dynamic Simulator: 정상 작동
- Metrics Calculator: 정상 작동
- Optimizer: 정상 작동
- UI Components: 정상 작동